# Maxwell photonic device

The geometry, Q2 local fields and P1 trace dimensions follow Lanteri et al. (2018), Section 6.3. The incident transient is explicitly defined in the case documentation. A classical central-DG sequence provides a numerical baseline, with separate space, time and quadrature controls. It is not an exact solution.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import MaxwellStepper
from pymhm.quadrilateral import CartesianMacroMesh
root = Path.cwd()
data = root / "examples/results/maxwell-nanoguide"

## Small operator check

A constant magnetic field in a PEC cavity has zero curl. Its electric field remains zero and the physical field is stationary. This checks the Cartesian path cheaply; the photonic comparison uses the independently acquired full fields.

In [ ]:
with threadpool_limits(1), MaxwellStepper(CartesianMacroMesh(2), time_step=0.005, degree=2, local_refinement=2) as stepper:
    first = stepper.initialize(magnetic=np.array([0.2, -0.3]))
    for _ in range(8):
        last = stepper.advance()
    assert abs(last.energy - first.energy) < 1e-12
    assert max(np.max(np.abs(e)) for e in last.electric) < 1e-12
print("Cartesian Q2 stationary-field check passed")

In [ ]:
import hashlib
record = json.loads((data / "comparison.json").read_text())
controls = json.loads((data / "refinement-controls.json").read_text())
verified = set()
for row in [*record["comparisons"], *controls.values()]:
    for key in ("field", "reference"):
        metadata = row[key]
        if metadata["file"] not in verified:
            assert hashlib.sha256((data / metadata["file"]).read_bytes()).hexdigest() == metadata["sha256"]
            verified.add(metadata["file"])
for row in record["comparisons"]:
    print(row["field"]["file"], "relative component L2:", row["relative_l2"], "combined:", row["combined_relative_l2"])
for name, row in controls.items():
    print(name, "combined relative increment:", row["combined_relative_l2"])
print("All field digests verified")


In [ ]:
for name in ["material", "components", "errors", "profiles", "published-electric", "published-coarse"]:
    display(Image(filename=str(root / "docs/figures/maxwell-nanoguide" / (name + ".png"))))

Spatial fields use the same component scales. Macro boundaries remain visible; profiles retain independent fine-cell traces. Printed article panels are attributed to Figures 6.9–6.10. Detailed phase/amplitude agreement requires the same initial and incident-wave conventions, which the article does not fully specify.